# KrishiDisha field disease classifier - Kaggle training

Trains a larger backbone (ConvNeXt-Tiny / EfficientNetV2-S) on the unified field manifest using a Kaggle GPU (T4 x2 or P100, 30 h/week).

**Setup on Kaggle** (Settings panel on the right):
1. Accelerator: GPU T4 x2 (or P100). Internet: ON (needed for `pip`, `git clone`, pretrained weights).
2. Add data: upload the pre-resized `disease_unified` folder from your machine as a private Kaggle dataset
   (zip `C:/Shivpratap_Singh_Official_Work/datasets/disease_unified` -> *Datasets > New Dataset*), then attach it here.
   Alternatively attach the raw Kaggle sources and run the manifest builder in this notebook (cell 4).
3. Set `REPO` to your GitHub repo URL and `BRANCH` if not `main`.

Outputs (checkpoint, ONNX, reports) are written to `/kaggle/working/models` - download them from the *Output* tab and copy into the repo's `models/` folder.

In [ ]:
# Source comes from the private Kaggle dataset (git archive of the lab repo), not from GitHub
import glob, os, shutil, zipfile
_src = (glob.glob("/kaggle/input/**/krishidisha-lab-src/src.zip", recursive=True) or glob.glob("/kaggle/input/**/src.zip", recursive=True))[0]
shutil.rmtree("/tmp/krishidisha", ignore_errors=True)
zipfile.ZipFile(_src).extractall("/tmp")
print("source:", _src, open(os.path.dirname(_src) + "/VERSION.txt").read().strip() if os.path.exists(os.path.dirname(_src) + "/VERSION.txt") else "")
ARCH = "timm:convnext_tiny"      # or timm:efficientnetv2_s, timm:efficientnet_b2
IMG_SIZE = 224
EPOCHS = 12
SMOKE = False                     # True: 1 short epoch on 30 images/class to validate the environment; set False for the real run
BATCH = 64

import os, subprocess, sys
os.chdir("/tmp/krishidisha")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "onnx", "onnxruntime", "pyyaml"], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# Locate the attached unified dataset (a folder containing manifest.csv + images/)
import glob
cands = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)
print(cands)
MANIFEST = cands[0] if cands else None
assert MANIFEST, "attach the disease_unified dataset (or build it in the next cell)"

In [ ]:
# Kaggle input is read-only; the trainer only reads images, so we can use it in place.
# Copying to local disk is faster for many small files:
import shutil, pathlib
src = pathlib.Path(MANIFEST).parent
dst = pathlib.Path("/kaggle/working/disease_unified")
if not dst.exists():
    shutil.copytree(src, dst)
MANIFEST = str(dst / "manifest.csv")
print(MANIFEST, sum(1 for _ in open(MANIFEST)) - 1, "rows")

In [ ]:
# Pre-flight: every manifest path must exist (an old dataset version silently misses whole sources)
import csv, os, collections
rows = list(csv.DictReader(open(MANIFEST, encoding="utf-8")))
root = os.path.dirname(MANIFEST)
missing = collections.Counter(r["source"] for r in rows if not os.path.exists(os.path.join(root, r["path"])))
print(len(rows), "rows;", sum(missing.values()), "missing", dict(missing))
assert not missing, f"dataset version out of sync with manifest: {dict(missing)} - re-attach the latest dataset version"


In [ ]:
# (Optional) build the manifest here from raw attached sources instead of uploading the unified folder.
# Set KRISHIDISHA_DATA_ROOT to a folder that contains <source_name>/ dirs matching ml/datasets/sources.yaml.
# os.environ["KRISHIDISHA_DATA_ROOT"] = "/kaggle/working/datasets"
# !python -m ml.datasets.build_disease_manifest --out /kaggle/working/disease_unified --resize 320

In [ ]:
EXTRA = "--limit-per-class 30 --epochs 1" if SMOKE else f"--epochs {EPOCHS}"
CMD = f"python -m ml.vision.train --manifest {MANIFEST} --arch {ARCH} --img-size {IMG_SIZE} --batch-size {BATCH} {EXTRA} --ema --num-workers 4 --output /kaggle/working/models --resume --calibrate 2>&1 | tee /kaggle/working/train.log | tail -40"
print(CMD)
!{CMD}


In [ ]:
if not SMOKE:
    CMD = f"python -m ml.vision.train --manifest {MANIFEST} --arch {ARCH} --img-size {IMG_SIZE} --batch-size {BATCH} --epochs 4 --lr 1e-4 --own-weight 6 --ema --num-workers 4 --output /kaggle/working/models_stageB --init /kaggle/working/models/plant_disease_model.pt --calibrate 2>&1 | tee /kaggle/working/stageB.log | tail -20"
    !{CMD}


In [ ]:
OUTM = "/kaggle/working/models" if SMOKE else "/kaggle/working/models_stageB"
CMD = f"python -m ml.vision.eval --checkpoint {OUTM}/plant_disease_model.pt --manifest {MANIFEST} --output {OUTM} 2>&1 | tail -30"
!{CMD}
CMD = f"python -m ml.vision.export --checkpoint {OUTM}/plant_disease_model.pt --output {OUTM} 2>&1 | tail -5"
!{CMD}
import shutil; shutil.rmtree('/kaggle/working/disease_unified', ignore_errors=True); shutil.rmtree('/tmp/krishidisha/.git', ignore_errors=True)
print(open(f"{OUTM}/reports/disease_eval.md").read()[:3000])
